In [2]:
import pandas as pd

# 1. Read in your CSV (make sure path is correct)
df = pd.read_csv('Anthropic_Sonnet35_full_results.csv')

# 2. (Optional) if you only want test‐split rows
df = df[df['split'] == 'test']

# 3. Grab the model name from the 'llm' column
model_name = df['llm'].iloc[0]

# 4. Get a sorted list of subjects so our rows stay in a consistent order
subjects = sorted(df['subject'].unique())

# 5. Loop over each language and sum up token counts
for lang in sorted(df['language'].unique()):
    df_lang = df[df['language'] == lang]

    # 5a. Sum tokens by subject
    subj_tokens = df_lang.groupby('subject')['num_tokens'].sum()

    # 5b. Build a DataFrame and name its column after the model
    table = subj_tokens.to_frame(name=model_name)

    # 5c. Compute the overall total‐tokens for this language
    total_tokens = df_lang['num_tokens'].sum()
    table.loc[f"Total {lang}"] = total_tokens

    # 5d. Reorder so the Total row sits at the top
    table = table.reindex([f"Total {lang}"] + subjects)

    # 5e. Print out a nice Markdown table
    print(f"\n### Token Counts for {lang} ({model_name})\n")
    print(table.to_markdown())


### Token Counts for amh (Anthropic/claude-3-5-sonnet-20241022)

| subject                    |   Anthropic/claude-3-5-sonnet-20241022 |
|:---------------------------|---------------------------------------:|
| Total amh                  |                                  99508 |
| elementary_mathematics     |                                  12001 |
| global_facts               |                                  14439 |
| high_school_geography      |                                  15577 |
| high_school_microeconomics |                                  24535 |
| international_law          |                                  32956 |

### Token Counts for eng (Anthropic/claude-3-5-sonnet-20241022)

| subject                    |   Anthropic/claude-3-5-sonnet-20241022 |
|:---------------------------|---------------------------------------:|
| Total eng                  |                                  30367 |
| elementary_mathematics     |                                   5733 |
| gl

In [4]:
import pandas as pd
import glob

# 1. Read & combine all model CSVs
all_files = glob.glob('*_full_results.csv')
dfs = []
for fn in all_files:
    df = pd.read_csv(fn)
    # filter to test‐split if you only want evaluation tokens
    df = df[df['split']=='test']
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

# 2. Prepare lists
languages = sorted(df_all['language'].unique())
subjects  = sorted(df_all['subject'].unique())

# 3. Loop through each language
for lang in languages:
    df_lang = df_all[df_all['language'] == lang]
    
    # 3a. Total token count per model
    total_tok = df_lang.groupby('llm')['num_tokens'].sum()
    total_df  = pd.DataFrame(total_tok).T
    total_df.index = [f"Total {lang}"]
    
    # 3b. Per‐subject token count per model
    subj_tok = (df_lang
                .pivot_table(
                    index='subject',
                    columns='llm',
                    values='num_tokens',
                    aggfunc='sum',
                    fill_value=0
                )
               )
    
    # 3c. Combine rows and reorder so “Total” is first
    combined = pd.concat([total_df, subj_tok])
    combined = combined.reindex([f"Total {lang}"] + subjects)
    
    # 3d. Alphabetical column order
    combined = combined[sorted(combined.columns)]
    
    # 3e. Print Markdown table
    print(f"\n### Token Counts for {lang}\n")
    print(combined.to_markdown())


### Token Counts for amh

|                            |   Anthropic/claude-3-5-sonnet-20241022 |   CohereLabs/aya-23-35B |   Google/gemini-15-Pro-002 |   OpenAI/gpt-4o-2024-08-06 |   OpenAI/o1-preview-2024-09-12 |   Qwen/Qwen2.5-32B |   deepseek-ai/DeepSeek-R1 |   deepseek-ai/DeepSeek-V3-0324 |   meta-llama/Llama-3.1-405B |   microsoft/phi-4 |   mistralai/Pixtral-12B-2409 |
|:---------------------------|---------------------------------------:|------------------------:|---------------------------:|---------------------------:|-------------------------------:|-------------------:|--------------------------:|-------------------------------:|----------------------------:|------------------:|-----------------------------:|
| Total amh                  |                                  99508 |                  146228 |                      75154 |                     133085 |                         133085 |              97638 |                    138175 |                         138175 

In [7]:
import pandas as pd
import glob

# find all of your ‘…_full_results.csv’ files
files = glob.glob('*_full_results.csv')

# accumulate tokenizer values
tokenizers = set()
for fn in files:
    df = pd.read_csv(fn, usecols=['tokenizer'])
    tokenizers.update(df['tokenizer'].dropna().unique())

# print them sorted
print("Unique tokenizers across all CSVs:")
for tok in sorted(tokenizers):
    print("-", tok)

Unique tokenizers across all CSVs:
- CohereLabs/aya-23-35B
- Qwen/Qwen2.5-32B
- anthropic/claude-3.5-sonnet
- deepseek-ai/DeepSeek-R1
- deepseek-ai/DeepSeek-V3-0324
- google/gemini-1.5-pro-002
- meta-llama/Llama-3.1-405B
- microsoft/phi-4
- mistralai/Pixtral-12B-2409
- openai/gpt-4o/o200k_base
- openai/o1-preview/o200k_base*


In [1]:
import pandas as pd
import glob

# 1) Read & concatenate all of your *_full_results.csv
csv_files = glob.glob("*_full_results.csv")
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# 2) Make the core pivot: (subject, language) × llm → sum(num_tokens)
pivot = (
    df
    .pivot_table(
        index=["subject", "language"],
        columns="llm",
        values="num_tokens",
        aggfunc="sum"
    )
)

# 3) Compute per‐subject subtotals
#    This gives a DataFrame indexed by subject, columns=llm
subs = df.groupby(["subject","llm"])["num_tokens"].sum().unstack(fill_value=0)

#  Build a MultiIndex for “(subject, All languages)”
sub_idx = pd.MultiIndex.from_product(
    [subs.index, ["All languages"]],
    names=["subject","language"]
)

subtotals = pd.DataFrame(
    data=subs.values,
    index=sub_idx,
    columns=subs.columns
)

# 4) Compute the grand total across every subject & language
grand = df.groupby("llm")["num_tokens"].sum()
grand_idx = pd.MultiIndex.from_tuples(
    [("All subjects", "")],
    names=["subject","language"]
)
grand_total = pd.DataFrame([grand.values], index=grand_idx, columns=grand.index)

# 5) Stitch them in order: for each subject, the languages then its subtotal,
#    then finally the overall total
out_frames = []
for subj in pivot.index.get_level_values("subject").unique():
    out_frames.append(pivot.loc[[subj]])                      # all (subj, lang) rows
    out_frames.append(subtotals.loc[subj:subj])               # the one subtotal row

out_frames.append(grand_total)

final = pd.concat(out_frames)

# 6) View or export
final

llm                                   Anthropic/claude-3-5-sonnet-20241022  \
subject                language                                              
elementary_mathematics amh                                           12001   
                       eng                                            5733   
                       ewe                                            9326   
                       fra                                            7131   
                       hau                                            7987   
...                                                                    ...   
international_law      xho                                           19209   
                       yor                                           31566   
                       zul                                           18692   
                       All languages                                373958   
All subjects                                                       1159265   

llm                                   CohereLabs/aya-23-35B  \
subject                language                               
elementary_mathematics amh                            16304   
                       eng                             5229   
                       ewe                             7734   
                       fra                             5812   
                       hau                             7050   
...                                                     ...   
international_law      xho                            16345   
                       yor                            22277   
                       zul                            15745   
                       All languages                 329358   
All subjects                                        1018523   

llm                                   Google/gemini-15-Pro-002  \
subject                language                                  
elementary_mathematics amh                                9559   
                       eng                                5206   
                       ewe                                7390   
                       fra                                5775   
                       hau                                6771   
...                                                        ...   
international_law      xho                               16249   
                       yor                               20942   
                       zul                               15432   
                       All languages                    292721   
All subjects                                            915028   

llm                                   OpenAI/gpt-4o-2024-08-06  \
subject                language                                  
elementary_mathematics amh                               14323   
                       eng                                4493   
                       ewe                                6951   
                       fra                                5078   
                       hau                                5753   
...                                                        ...   
international_law      xho                               13572   
                       yor                               17762   
                       zul                               12796   
                       All languages                    286468   
All subjects                                            876352   

llm                                   OpenAI/o1-preview-2024-09-12  \
subject                language                                      
elementary_mathematics amh                                   14323   
                       eng                                    4493   
                       ewe                                    6951   
                       fra                                    5078   
                       hau 

In [5]:
import pandas as pd

# 1) Lift the display limits:
pd.set_option("display.max_rows",    None)   # show all rows
pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width",       None)   # don’t wrap lines

# 2) If you already have `final` from the pivot/concat step:
display(final)  

llm                                       Anthropic/claude-3-5-sonnet-20241022  \
subject                    language                                              
elementary_mathematics     amh                                           12001   
                           eng                                            5733   
                           ewe                                            9326   
                           fra                                            7131   
                           hau                                            7987   
                           ibo                                            9785   
                           kin                                            7781   
                           lin                                            7373   
                           lug                                            8886   
                           orm                                            8582   
                           sna                                            7597   
                           sot                                            8098   
                           swa                                            7905   
                           twi                                           10489   
                           wol                                            8070   
                           xho                                            8331   
                           yor                                           11774   
                           zul                                            8497   
                           All languages                                155346   
global_facts               amh                                           14439   
                           eng                                            4913   
                           ewe                                           11820   
                           fra                                            7133   
                           hau                                            8140   
                           ibo                                            9673   
                           kin                                            9079   
                           lin                                            7445   
                           lug                                            9762   
                           orm                                            9172   
                           sna                                            8396   
                           sot                                            8378   
                           swa                                            8134   
                           twi                                           10565   
                           wol                                            9249   
                           xho                                            9286   
                           yor                                           14173   
                           zul                                            8710   
                           All languages                                168467   
high_school_geography      amh                                           15577   
                           eng                                            4295   
                           ewe                                           10783   
                           fra                                            7251   
                           hau                                            9047   
                           ibo                                           10222   
                           kin                                            9751   
                           lin                                            7672   
                           lug                                  

In [7]:
import pandas as pd
import glob

# 1) Read & concatenate
csv_files = glob.glob("*_full_results.csv")
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# 2) Pivot into a MultiIndex of (subject, language) × llm → sum(num_tokens)
final = (
    df
    .pivot_table(
        index=["subject", "language"],
        columns="llm",
        values="num_tokens",
        aggfunc="sum"
    )
    .sort_index()
)

# 3) (Optional) lift pandas display limits in Jupyter
pd.set_option("display.max_rows",    None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width",       None)

# 4) Show it
display(final)

llm                                  Anthropic/claude-3-5-sonnet-20241022  \
subject                    language                                         
elementary_mathematics     amh                                      12001   
                           eng                                       5733   
                           ewe                                       9326   
                           fra                                       7131   
                           hau                                       7987   
                           ibo                                       9785   
                           kin                                       7781   
                           lin                                       7373   
                           lug                                       8886   
                           orm                                       8582   
                           sna                                       7597   
                           sot                                       8098   
                           swa                                       7905   
                           twi                                      10489   
                           wol                                       8070   
                           xho                                       8331   
                           yor                                      11774   
                           zul                                       8497   
global_facts               amh                                      14439   
                           eng                                       4913   
                           ewe                                      11820   
                           fra                                       7133   
                           hau                                       8140   
                           ibo                                       9673   
                           kin                                       9079   
                           lin                                       7445   
                           lug                                       9762   
                           orm                                       9172   
                           sna                                       8396   
                           sot                                       8378   
                           swa                                       8134   
                           twi                                      10565   
                           wol                                       9249   
                           xho                                       9286   
                           yor                                      14173   
                           zul                                       8710   
high_school_geography      amh                                      15577   
                           eng                                       4295   
                           ewe                                      10783   
                           fra                                       7251   
                           hau                                       9047   
                           ibo                                      10222   
                           kin                                       9751   
                           lin                                       7672   
                           lug                                      11125   
                           orm                                      10262   
                           sna                                       8381   
                           sot                                       7751   
                           swa                                       8711   
                           twi                                      13172

In [13]:
final.to_csv("token_counts_by_subject_language.csv")

In [11]:
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# 3) Pivot so rows are 'subject' and columns are each model ('llm'),
#    with the sum of 'num_tokens' as the values
subject_vs_llm = (
    df
    .pivot_table(
        index="subject",
        columns="llm",
        values="num_tokens",
        aggfunc="sum"
    )
    .sort_index()
)

# 4) Inspect it
subject_vs_llm

llm,Anthropic/claude-3-5-sonnet-20241022,CohereLabs/aya-23-35B,Google/gemini-15-Pro-002,OpenAI/gpt-4o-2024-08-06,OpenAI/o1-preview-2024-09-12,Qwen/Qwen2.5-32B,deepseek-ai/DeepSeek-R1,deepseek-ai/DeepSeek-V3-0324,meta-llama/Llama-3.1-405B,microsoft/phi-4,mistralai/Pixtral-12B-2409
subject,,,,,,,,,,,
elementary_mathematics,155346,139818,129890,116916,116916,139378,129987,129987,134886,136398,150502
global_facts,168467,151660,138066,126612,126612,152182,146670,146670,153473,156296,164890
high_school_geography,177291,148527,132774,131205,131205,148592,154270,154270,163635,166514,165184
high_school_microeconomics,284203,249160,221577,215151,215151,248584,258654,258654,271328,276110,271734
international_law,373958,329358,292721,286468,286468,328851,342758,342758,361698,368050,360474


In [12]:
subject_vs_llm.to_csv("subject_vs_llm_token_counts.csv")